In [ ]:
import json, subprocess, sys
from pathlib import Path
from google.cloud import storage

PROJECT, BUCKET = "dark-data-discovery", "dark-data-discovery-fsl-data"
REPO = Path.home() / "mlops-meta-learning/fsl-projekt"
OUT = Path.home() / "explain_reports"
OUT.mkdir(exist_ok=True)
N_EPISODES = 200

# czy env widzi NOWY samld.py (swiezy interpreter, bez pulapki starego modulu)
chk = subprocess.run([sys.executable, "-c",
                      "from fsl.data.samld import GroupedEpisodeSampler as G; "
                      "assert hasattr(G, 'sample_indexed')"], capture_output=True, text=True)
assert chk.returncode == 0, ("env nie widzi nowego samld.py -> w terminalu: "
                             "cd ~/mlops-meta-learning/fsl-projekt && ~/envs/fsl/bin/pip install -e .\n"
                             + chk.stderr[-400:])

# najnowszy artefakt samld per fold
client = storage.Client(project=PROJECT)
latest = {}
for b in client.list_blobs(BUCKET, prefix="models/"):
    if not b.name.endswith("architecture.json"):
        continue
    arch = json.loads(b.download_as_text())
    if arch.get("dataset") != "samld":
        continue
    k = int(arch.get("fold", 0))
    if k not in latest or b.updated > latest[k][0]:
        latest[k] = (b.updated, f"gs://{BUCKET}/{b.name.rsplit('/', 1)[0]}")
assert latest, f"brak artefaktow samld pod gs://{BUCKET}/models/"
for k in sorted(latest):
    print(f"fold {k}: {latest[k][1]}   ({latest[k][0]:%Y-%m-%d %H:%M})")

# explain per fold, fold 1 pierwszy
for k in sorted(latest, key=lambda f: (f != 1, f)):
    cmd = [sys.executable, str(REPO / "scripts/explain_pipeline_entry.py"),
           "--project", PROJECT, "--bucket", BUCKET, "--dataset", "samld",
           "--model-dir", latest[k][1], "--model-version", f"fold{k}",
           "--n-episodes", str(N_EPISODES),
           "--report-html-path", str(OUT / f"fold{k}.html"),
           "--summary-output-path", str(OUT / f"fold{k}.txt"),
           "--json-output-path", str(OUT / f"fold{k}.json")]
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO)
    print(f"\nfold {k}:", r.stdout.strip().splitlines()[-1] if r.returncode == 0
          else "FAILED\n" + r.stderr[-2000:])

In [ ]:
import html
import pandas as pd
from IPython.display import HTML, display

rows = []
for k in sorted(latest):
    p = OUT / f"fold{k}.json"
    if not p.exists():
        continue
    s = json.loads(p.read_text())
    t, tc = s["tests"], (s["top_confusions"] or [{}])[0]
    rows.append({
        "fold": k, "acc": round(s["accuracy"], 3),
        "completeness p95": f"{t['completeness']['p95_rel_residual']:.2%}" + (" OK" if t["completeness"]["passed"] else " FAIL"),
        "faithfulness": f"{t['faithfulness']['ratio']:.1f}x" + (" OK" if t["faithfulness"]["passed"] else " FAIL"),
        "flips top/rand": f"{t['faithfulness']['flip_rate_top']:.0%} / {t['faithfulness']['flip_rate_random']:.0%}",
        "stability rho": f"{t['stability']['median_spearman']:.2f}" + (" OK" if t["stability"]["passed"] else " FAIL"),
        "same top family": f"{t['stability']['top_family_agreement']:.0%}",
        "top confusion": f"{tc.get('true', '-')} -> {tc.get('pred', '-')} ({tc.get('rate', 0):.0%})",
    })
display(pd.DataFrame(rows).set_index("fold"))

doc = (OUT / "fold1.html").read_text()
display(HTML(f'<iframe srcdoc="{html.escape(doc)}" '
             f'style="width:100%;height:1100px;border:1px solid #ddd;border-radius:8px"></iframe>'))